# 🎙️➡️🎨 From Voice to Vision — Fase 4: Ottimizzazione Bio-Ispirata

Il cuore del progetto. Ottimizziamo gli iperparametri della CNN con **tre algoritmi bio-ispirati**
e li mettiamo a confronto:
- **PSO** — Particle Swarm Optimization
- **FSO** — Flock of Starlings Optimization *(l'ingrediente originale: allineamento con i vicini topologici, come gli storni)*
- **GA** — Genetic Algorithm (real-coded)

Spazio di ricerca: `lr`, `dropout`, `weight_decay`, `batch_size`, `width` della CNN.
Fitness = **validation accuracy** di una CNN allenata brevemente per ogni candidato.

⏱️ **Nota tempi:** ogni algoritmo allena molte CNN brevi. Con i valori di default
(6 agenti × 6 iterazioni ≈ 42 addestramenti) ognuno richiede ~10-15 min su GPU T4.
Puoi lanciare i tre algoritmi anche in momenti diversi. Per una prova rapida usa `QUICK = True`.

In [ ]:
# === 1. Repo + librerie + feature ===
REPO_URL = "https://github.com/<tuo-username>/from-voice-to-vision.git"
import os
repo = REPO_URL.rstrip("/").split("/")[-1].replace(".git","")
if not os.path.exists(repo):
    !git clone $REPO_URL
%cd $repo
!git pull -q
!pip install -q librosa soundfile noisereduce tqdm

from src import config, data_loader, features
data_loader.download_ravdess()
df = data_loader.build_index()
data = features.build_dataset(df, denoise=False, cache=True)
splits = features.split_arrays(data)
print("Pronto. Train/val/test:", [len(splits[s]['y']) for s in ['train','val','test']])

In [ ]:
# === 2. Budget di ricerca (regola qui) ===
QUICK = False              # True = prova veloce (5 agenti x 3 iter, 8 epoche)
if QUICK:
    N_AGENTS, N_ITER, EPOCHS_SEARCH = 5, 3, 8
else:
    N_AGENTS, N_ITER, EPOCHS_SEARCH = 6, 6, 12

from src.optimization import pso, fso, ga
from src.optimization.search_space import make_objective, decode, DIM
import time

def run(optimizer, **kw):
    obj = make_objective(splits, epochs=EPOCHS_SEARCH, patience=4)
    t0 = time.time()
    res = optimizer.optimize(obj, DIM, seed=config.SEED, **kw)
    res["time_s"] = time.time() - t0
    res["n_evals"] = len(obj.history_evals)
    res["best_hp"] = decode(res["best_u"])
    print(f"→ {res['name']}: best_val={res['best_fit']:.3f} | valutazioni={res['n_evals']} | tempo={res['time_s']:.0f}s")
    return res

### PSO

In [ ]:
res_pso = run(pso, n_particles=N_AGENTS, n_iter=N_ITER)
print(res_pso["best_hp"])

### FSO (Flock of Starlings)

In [ ]:
res_fso = run(fso, n_particles=N_AGENTS, n_iter=N_ITER)
print(res_fso["best_hp"])

### GA

In [ ]:
res_ga = run(ga, pop_size=N_AGENTS, n_iter=N_ITER)
print(res_ga["best_hp"])

## Confronto: convergenza ed efficienza

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

allres = [res_pso, res_fso, res_ga]
plt.figure(figsize=(9,5))
for r in allres:
    plt.plot(range(len(r["history"])), r["history"], marker="o", label=r["name"])
plt.xlabel("iterazione"); plt.ylabel("miglior validation accuracy")
plt.title("Convergenza: PSO vs FSO vs GA"); plt.legend(); plt.grid(alpha=.3)
plt.tight_layout(); plt.savefig(config.FIGURES_DIR / "03_convergence.png", dpi=120); plt.show()

tab = pd.DataFrame([{
    "algoritmo": r["name"], "best_val_acc": round(r["best_fit"],3),
    "valutazioni": r["n_evals"], "tempo_s": round(r["time_s"]),
    **r["best_hp"]
} for r in allres])
display(tab)

## Modello finale: riaddestramento della configurazione migliore

In [ ]:
from src.models import cnn
import json

best = max(allres, key=lambda r: r["best_fit"])
print(f"Vince: {best['name']}  con val_acc={best['best_fit']:.3f}")
print("Iperparametri:", best["best_hp"])

# training completo (più epoche) della config migliore
final = cnn.train_cnn(splits, best["best_hp"], epochs=60, patience=12, verbose=True)
print(f"\n*** CNN ottimizzata — test accuracy = {final['test_acc']:.3f} ***")
print(final["report"])

In [ ]:
# Matrice di confusione del modello finale + salvataggi
import seaborn as sns
plt.figure(figsize=(7,6))
sns.heatmap(final["cm"], annot=True, fmt="d", cmap="Blues", cbar=False,
            xticklabels=config.EMOTIONS, yticklabels=config.EMOTIONS)
plt.title(f"CNN ottimizzata ({best['name']}) — test={final['test_acc']:.2f}")
plt.xlabel("predetto"); plt.ylabel("reale"); plt.xticks(rotation=45)
plt.tight_layout(); plt.savefig(config.FIGURES_DIR / "03_confusion_optimized.png", dpi=120); plt.show()

config.RESULTS_DIR.mkdir(parents=True, exist_ok=True)
with open(config.RESULTS_DIR / "best_hp.json", "w") as f:
    json.dump({"winner": best["name"], "hp": best["best_hp"],
               "val_acc": best["best_fit"], "test_acc": final["test_acc"]}, f, indent=2)
with open(config.RESULTS_DIR / "optimization_results.json", "w") as f:
    json.dump([{ "name": r["name"], "best_val_acc": r["best_fit"],
                 "n_evals": r["n_evals"], "time_s": r["time_s"],
                 "history": r["history"], "hp": r["best_hp"]} for r in allres], f, indent=2)
# salva il modello per le fasi XAI (Grad-CAM) e arte
import torch
torch.save({"state_dict": final["model"].state_dict(), "hp": best["best_hp"],
            "norm": final["norm"]}, config.RESULTS_DIR / "best_cnn.pt")
print("✓ salvati best_hp.json, optimization_results.json, best_cnn.pt")

### ✅ Fase 4 completata
Abbiamo confrontato PSO, FSO e GA e riaddestrato la configurazione migliore.

**Mandami:** la tabella di confronto, la curva di convergenza e l'accuracy finale della CNN ottimizzata.
Poi in **Fase 6** apriamo la "scatola nera" con la **explainability** (Grad-CAM sugli spettrogrammi, t-SNE, SHAP).